# 1. Config

In [ ]:
import os
import pandas as pd

from pathlib import Path

from dotenv import load_dotenv
import polars as pl

import seaborn as sns
import matplotlib.pyplot as plt
import altair as alt
from datetime import datetime
import random

DATASET_DIR = Path(__file__).parent.parent / 'datasets' / 'otto-recommender-system'
TRAIN_JSON = DATASET_DIR / 'train.jsonl'
TEST_JSON = DATASET_DIR / 'test.jsonl'

# Convert JSONL to parquet

In [ ]:
# OUTPUT_PATH = Path('./datasets/otto-recommender-system')
# TRAIN_PARQUET = OUTPUT_PATH / 'train.parquet'
# TEST_PARQUET = OUTPUT_PATH / 'test.parquet'

In [3]:
# def json_to_parquet(file_path, output_path):
# 	df = pl.scan_ndjson(file_path) \
#     .sink_parquet(output_path, compression='zstd', compression_level=3)

# json_to_parquet(
# 	str(TRAIN_JSON),
# 	str(TRAIN_PARQUET)
# )

# json_to_parquet(
# 	str(TEST_JSON), str(TEST_PARQUET)
# )

# 1. EDA

In [ ]:
PARQUET_DIR = Path(__file__).parent.parent / 'datasets' / 'otto-parquet'
TRAIN_PARQUET = PARQUET_DIR / 'train.parquet'
TEST_PARQUET = PARQUET_DIR / 'test.parquet'

In [5]:
train_df = pl.scan_parquet(TRAIN_PARQUET)
test_df = pl.scan_parquet(TEST_PARQUET)

## 1.1. Xem dữ liệu train và test

In [6]:
# schema
train_df.schema

/tmp/ipykernel_55/2186414520.py:2: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  train_df.schema


Schema([('session', Int64),
        ('events', List(Struct({'aid': Int64, 'ts': Int64, 'type': String})))])

In [7]:
# Xem một vài dữ liệu đầu tiên trong tập train và tập test
print('Tập train:')
print(train_df.head(3).collect())

print('-'*50)

print('Tập test:')
print(test_df.head(3).collect())

Tập train:
shape: (3, 2)
┌─────────┬─────────────────────────────────┐
│ session ┆ events                          │
│ ---     ┆ ---                             │
│ i64     ┆ list[struct[3]]                 │
╞═════════╪═════════════════════════════════╡
│ 0       ┆ [{1517085,1659304800025,"click… │
│ 1       ┆ [{424964,1659304800025,"carts"… │
│ 2       ┆ [{763743,1659304800038,"clicks… │
└─────────┴─────────────────────────────────┘
--------------------------------------------------
Tập test:
shape: (3, 2)
┌──────────┬─────────────────────────────────┐
│ session  ┆ events                          │
│ ---      ┆ ---                             │
│ i64      ┆ list[struct[3]]                 │
╞══════════╪═════════════════════════════════╡
│ 12899779 ┆ [{59625,1661724000278,"clicks"… │
│ 12899780 ┆ [{1142000,1661724000378,"click… │
│ 12899781 ┆ [{141736,1661724000559,"clicks… │
└──────────┴─────────────────────────────────┘


In [8]:
train_df.head(1).collect() \
    .explode("events") \
    .unnest("events")

session,aid,ts,type
i64,i64,i64,str
0,1517085,1659304800025,"""clicks"""
0,1563459,1659304904511,"""clicks"""
0,1309446,1659367439426,"""clicks"""
0,16246,1659367719997,"""clicks"""
0,1781822,1659367871344,"""clicks"""
…,…,…,…
0,843110,1661684298768,"""clicks"""
0,938007,1661684355390,"""clicks"""
0,1228848,1661684528943,"""clicks"""


In [9]:
test_df.head(1).collect().explode('events').unnest('events')

session,aid,ts,type
i64,i64,i64,str
12899779,59625,1661724000278,"""clicks"""


## 1.2. Thống kê đơn giản

In [10]:
def get_events_stats(df, name):
    n_events_per_session = df.collect().select(pl.col('events').list.len().alias('n_events'))['n_events']
    return {
        'Dataset': name,
        'mean': f'{n_events_per_session.mean():.2f}',
        'std': f'{n_events_per_session.std():.2f}',
        'min': n_events_per_session.min(),
        '50%': n_events_per_session.quantile(0.5),
        '75%': n_events_per_session.quantile(0.75),
        '90%': n_events_per_session.quantile(0.9),
        '95%': n_events_per_session.quantile(0.95),
        'max': n_events_per_session.max(),
    }

In [11]:
print('Thống kê về số events của từng phiên trong tập train')

print(get_events_stats(train_df, 'train'))

print('-'*50)
print('Thống kê về số events của từng phiên trong tập test')

print(get_events_stats(test_df, 'test'))

Thống kê về số events của từng phiên trong tập train
{'Dataset': 'train', 'mean': '16.80', 'std': '33.58', 'min': 2, '50%': 6.0, '75%': 15.0, '90%': 39.0, '95%': 68.0, 'max': 500}
--------------------------------------------------
Thống kê về số events của từng phiên trong tập test
{'Dataset': 'test', 'mean': '4.14', 'std': '8.22', 'min': 1, '50%': 2.0, '75%': 4.0, '90%': 9.0, '95%': 15.0, 'max': 458}


In [12]:
# Đếm số session, items, events, clicks, carts, orders, và Density
def dataset_stats(df, name):
    df = df.collect()
    n_session = df.height
    
    events = df.select(pl.col('events').list.explode().alias('event')).unnest('event')
    n_events = events.height
    n_items = events.select(pl.col('aid')).unique().height

    type_counts_df = events.group_by('type').count()
    type_dict = dict(zip(type_counts_df['type'], type_counts_df['count']))

    n_clicks = type_dict.get('clicks', 0)
    n_carts = type_dict.get('carts', 0)
    n_orders = type_dict.get('orders', 0)

    density = (n_events / (n_session * n_items) * 100) if (n_session * n_items) > 0 else 0

    return {
        'Dataset': name,
        '#sessions': n_session,
        '#items': n_items,
        '#events': n_events,
        '#clicks': n_clicks,
        '#carts': n_carts,
        '#orders': n_orders,
        'Density': density
    }

In [13]:
print('Thống kê dataset trong tập train:')

print(dataset_stats(train_df, 'train'))

print('-'*50)
print('Thống kê dataset trong tập test:')
print(dataset_stats(test_df, 'test'))

Thống kê dataset trong tập train:


/tmp/ipykernel_55/2943913971.py:10: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  type_counts_df = events.group_by('type').count()


{'Dataset': 'train', '#sessions': 12899779, '#items': 1855603, '#events': 216716096, '#clicks': 194720954, '#carts': 16896191, '#orders': 5098951, 'Density': 0.0009053652736086677}
--------------------------------------------------
Thống kê dataset trong tập test:
{'Dataset': 'test', '#sessions': 1671803, '#items': 783486, '#events': 6928123, '#clicks': 6292632, '#carts': 570011, '#orders': 65480, 'Density': 0.0005289312769979806}


## 1.3. Số lượng các loại hành vi trong tập huấn luyện và tập kiểm tra

In [14]:
def count_event_types(df, name):
    return df.collect() \
            .explode('events') \
            .unnest('events') \
            .group_by('type') \
            .count() \
            .sort('count', descending=True)

In [15]:
train_events = count_event_types(train_df, 'train')
test_events = count_event_types(test_df, 'test')

/tmp/ipykernel_55/3517702589.py:6: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count() \


In [16]:
print('Số lượng event type trong tập train:')
print(train_events)

print('-'*50)
print('Số lượng event type trong tập test:')
print(test_events)

Số lượng event type trong tập train:
shape: (3, 2)
┌────────┬───────────┐
│ type   ┆ count     │
│ ---    ┆ ---       │
│ str    ┆ u32       │
╞════════╪═══════════╡
│ clicks ┆ 194720954 │
│ carts  ┆ 16896191  │
│ orders ┆ 5098951   │
└────────┴───────────┘
--------------------------------------------------
Số lượng event type trong tập test:
shape: (3, 2)
┌────────┬─────────┐
│ type   ┆ count   │
│ ---    ┆ ---     │
│ str    ┆ u32     │
╞════════╪═════════╡
│ clicks ┆ 6292632 │
│ carts  ┆ 570011  │
│ orders ┆ 65480   │
└────────┴─────────┘


In [ ]:
def make_chart(df, title, color):
    return alt.Chart(df).mark_bar(color=color).encode(
        x=alt.X('type:N', title='Loại sự kiện'),
        y=alt.Y('count:Q', title='Số lượng'),
        tooltip=['type', 'count']
    ).properties(
        width=500,
        height=400,
        title=title
    )

chart_train = make_chart(train_events, 'Biểu đồ số lượng các loại hành vi trong tập train', '#66c2a5')
chart_test  = make_chart(test_events,  'Biểu đồ số lượng các loại hành vi trong tập test',  '#fc8d62')

(chart_train | chart_test).show()

alt.HConcatChart(...)

## 1.4. Xét phân phối longtail của sản phẩm trong tập train

In [18]:
item_counts = train_df.collect() \
                .explode('events') \
                .unnest('events') \
                .group_by('aid') \
                .agg(pl.len().alias('count')) \
                .sort('count', descending=True) \
                .with_row_index('rank')

longtail_chart = alt.Chart(item_counts.head(5000)).mark_area(
    line={'strokeWidth': 2.5},
    opacity=0.4
).encode(
    x=alt.X('rank:Q', title='Xếp hạng sản phẩm (Item rank)'),
    y=alt.Y('count:Q', title='Số lượng tương tác'),
    tooltip=['aid', 'count', 'rank']
).properties(
    width=700, height=400,
    title=alt.TitleParams(
        text='Số lượng tương tác theo sản phẩm trong tập train (top 5000 item)',
        fontSize=16
    )
).configure_axis(
    labelFontSize=12,
    titleFontSize=13
)

longtail_chart.show()

alt.Chart(...)

## 1.5. Xét hành vi người dùng theo khung giờ trong ngày

In [19]:
hourly_counts = train_df.collect() \
                    .explode('events') \
                    .unnest('events') \
                    .with_columns(
                        pl.from_epoch('ts', time_unit='ms').dt.hour().alias('hour')
                    ) \
                    .group_by(['hour', 'type']) \
                    .agg(pl.len().alias('count')) \
                    .sort('hour')

In [20]:
chart = alt.Chart(hourly_counts).mark_line(strokeWidth=2.5, point=True).encode(
    x=alt.X('hour:O', title='Giờ trong ngày'),
    y=alt.Y('count:Q', title='Số lượng hành vi'),
    color=alt.Color('type:N', title='Loại hành vi'),
    tooltip=['hour', 'type', 'count']
).properties(
    width=700,
    height=400,
    title=alt.TitleParams(
        text='Hành vi người dùng theo khung giờ trong ngày (Train)',
        fontSize=16,
    )
).configure_axis(
    labelFontSize=12,
    titleFontSize=13
)

chart.show()

alt.Chart(...)

# 2. Create validation set

In [ ]:
OUTPUT_DIR = Path(__file__).parent.parent / 'datasets' / 'otto-train-val'
TRAIN_PARQUET = Path(__file__).parent.parent / 'datasets' / 'otto-parquet' / 'train.parquet'
TRAIN_SESSIONS_PARQUET = OUTPUT_DIR / 'train_sessions.parquet'
VALID_PARQUET = OUTPUT_DIR / 'valid_inputs.parquet'
VALID_LABELS_PARQUET = OUTPUT_DIR / 'valid_labels.parquet'

In [ ]:
import polars as pl
from pathlib import Path
import psutil
import random
from copy import deepcopy
from tqdm.auto import tqdm


def _ground_truth(events: list[dict]):
    prev_labels = {"clicks": None, "carts": set(), "orders": set()}
    for event in reversed(events):
        event["labels"] = {}
        for label in ['clicks', 'carts', 'orders']:
            if prev_labels[label]:
                if label != 'clicks':
                    event["labels"][label] = prev_labels[label].copy()
                else:
                    event["labels"][label] = prev_labels[label]
        if event["type"] == "clicks":
            prev_labels['clicks'] = event["aid"]
        elif event["type"] == "carts":
            prev_labels['carts'].add(event["aid"])
        elif event["type"] == "orders":
            prev_labels['orders'].add(event["aid"])
    return events[:-1]


def check_ram_available(min_gb=2):
    available = psutil.virtual_memory().available / (1024**3)
    total = psutil.virtual_memory().total / (1024**3)
    print(f"\nRAM CHECK:")
    print(f"  Total:     {total:.1f} GB")
    print(f"  Available: {available:.1f} GB")
    if available < min_gb:
        print(f"  WARNING: chỉ còn {available:.1f}GB (cần >= {min_gb}GB)")
        return False
    print(f"  OK")
    return True


def simple_split_parquet(train_parquet_path, output_dir, test_days=7, seed=42):
    random.seed(seed)
    check_ram_available(min_gb=2)

    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    print(f"\n{'='*60}")
    print("OTTO TRAIN/VALID SPLIT")
    print(f"{'='*60}")

    # Bước 1: Tính split_ts
    print("\n[1/6] Tính split timestamp...")
    events_df = pl.scan_parquet(train_parquet_path).explode('events').unnest('events')

    max_ts = events_df.select(pl.col('ts').max()).collect().item()
    split_ts = max_ts - test_days * 24 * 60 * 60 * 1000
    print(f"      max_ts   = {max_ts}")
    print(f"      split_ts = {split_ts}")
    print(f"      test period = {test_days} days")

    # Bước 2: Chia session train/test
    print("\n[2/6] Phân loại sessions...")
    session_first_ts = (
        events_df
        .group_by('session')
        .agg(pl.col('ts').min().alias('first_ts'))
    )

    train_session_ids_df = session_first_ts.filter(pl.col('first_ts') <= split_ts).select('session')
    test_session_ids_df  = session_first_ts.filter(pl.col('first_ts') >  split_ts).select('session')

    n_train_sessions = train_session_ids_df.collect().height
    n_test_sessions  = test_session_ids_df.collect().height
    print(f"      Train sessions: {n_train_sessions:,}")
    print(f"      Test sessions:  {n_test_sessions:,}")

    # Bước 3: Tạo train_sessions
    print("\n[3/6] Tạo train sessions (trim events sau split_ts)...")
    with tqdm(total=1, desc="      Building train sessions") as pbar:
        train_sessions = (
            events_df
            .join(train_session_ids_df, on='session', how='semi')
            .filter(pl.col('ts') < split_ts)
            .group_by('session')
            .agg(
                pl.struct(['aid', 'ts', 'type'])
                .sort_by('ts')
                .alias('events')
            )
            .filter(pl.col('events').list.len() >= 2)
            .collect()
        )
        pbar.update(1)
    print(f"      Train sessions sau filter: {train_sessions.height:,}")

    # Bước 4: Lọc unknown items trong test
    print("\n[4/6] Filter unknown items trong test sessions...")
    
    with tqdm(total=1, desc="      Collecting train aids") as pbar:
        train_items_df = (
            events_df
            .join(train_session_ids_df, on='session', how='semi')
            .filter(pl.col('ts') < split_ts)
            .select('aid')
            .unique()
            .collect()
        )
        train_aids_set = set(train_items_df['aid'].to_list())
        pbar.update(1)
    print(f"      Train unique aids: {len(train_aids_set):,}")
    
    with tqdm(total=1, desc="      Collecting test events") as pbar:
        test_events_raw = (
            events_df
            .join(test_session_ids_df, on='session', how='semi')
            .collect()
        )
        pbar.update(1)
    
    # Eager join sau collect
    with tqdm(total=1, desc="      Filtering & grouping test sessions") as pbar:
        test_sessions_filtered = (
            test_events_raw
            .join(train_items_df, on='aid', how='semi')
            .group_by('session')
            .agg(
                pl.struct(['aid', 'ts', 'type'])
                .sort_by('ts')
                .alias('events')
            )
            .filter(pl.col('events').list.len() >= 2)
        )
        pbar.update(1)
    
    print(f"      Test sessions sau filter: {test_sessions_filtered.height:,}")

    # Bước 5: Tạo valid input/label pairs
    print("\n[5/6] Tạo valid input/label pairs...")
    valid_inputs_list = []
    valid_labels_list = []
    skipped = 0

    for row in tqdm(test_sessions_filtered.iter_rows(named=True),
                    total=test_sessions_filtered.height,
                    desc="      Processing sessions"):
        session_id = row['session']
        events = [dict(e) for e in row['events']]

        if len(events) < 2:
            skipped += 1
            continue

        test_events = _ground_truth(deepcopy(events))
        if not test_events:
            skipped += 1
            continue

        split_idx = random.randint(1, len(test_events))
        test_events_trimmed = test_events[:split_idx]
        raw_labels = test_events_trimmed[-1]['labels']

        clean_clicks = raw_labels.get('clicks')
        if clean_clicks is not None and clean_clicks not in train_aids_set:
            clean_clicks = None

        clean_carts  = [a for a in raw_labels.get('carts',  set()) if a in train_aids_set]
        clean_orders = [a for a in raw_labels.get('orders', set()) if a in train_aids_set]

        if not any([clean_clicks, clean_carts, clean_orders]):
            skipped += 1
            continue

        history = [{k: v for k, v in e.items() if k != 'labels'}
                   for e in test_events_trimmed]

        valid_inputs_list.append({'session': session_id, 'events': history})
        valid_labels_list.append({
            'session': session_id,
            'labels': {
                'clicks': clean_clicks,
                'carts':  clean_carts,
                'orders': clean_orders,
            }
        })

    print(f"      Valid pairs tạo được: {len(valid_inputs_list):,}")
    print(f"      Skipped: {skipped:,}")

    # Bước 6: Ghi parquet
    print("\n[6/6] Ghi parquet...")

    files = {
        'train_sessions.parquet': train_sessions,
        'valid_inputs.parquet':   pl.DataFrame(valid_inputs_list),
        'valid_labels.parquet':   pl.DataFrame(valid_labels_list),
    }

    for fname, df in tqdm(files.items(), desc="      Writing files"):
        df.write_parquet(
            str(output_dir / fname),
            compression='zstd',
            compression_level=3
        )

    print(f"\n{'='*60}")
    print("DONE!")
    print(f"{'='*60}")
    print(f"\n  Train sessions : {train_sessions.height:,}")
    print(f"  Valid sessions : {len(valid_inputs_list):,}")
    print(f"\n  Files saved to: {output_dir}")
    for fname in files:
        path = output_dir / fname
        size_mb = path.stat().st_size / (1024**2)
        print(f"    {fname:<30} {size_mb:.1f} MB")


simple_split_parquet(
    train_parquet_path=TRAIN_PARQUET,
    output_dir=OUTPUT_DIR,
    test_days=7,
    seed=42
)


RAM CHECK:
  Total:     31.4 GB
  Available: 28.2 GB
  OK

OTTO TRAIN/VALID SPLIT

[1/6] Tính split timestamp...
      max_ts   = 1661723999984
      split_ts = 1661119199984
      test period = 7 days

[2/6] Phân loại sessions...
      Train sessions: 11,098,528
      Test sessions:  1,801,251

[3/6] Tạo train sessions (trim events sau split_ts)...


      Building train sessions:   0%|          | 0/1 [00:00<?, ?it/s]

      Train sessions sau filter: 10,584,517

[4/6] Filter unknown items trong test sessions...


      Train unique aids: 1,825,499


      Filtering & grouping test sessions:   0%|          | 0/1 [00:00<?, ?it/s]

      Test sessions sau filter: 1,783,790

[5/6] Tạo valid input/label pairs...


      Processing sessions:   0%|          | 0/1783790 [00:00<?, ?it/s]

      Valid pairs tạo được: 1,783,790
      Skipped: 0

[6/6] Ghi parquet...


      Writing files:   0%|          | 0/3 [00:00<?, ?it/s]


DONE!

  Train sessions : 10,584,517
  Valid sessions : 1,783,790

  Files saved to: /kaggle/working
    train_sessions.parquet         1044.1 MB
    valid_inputs.parquet           57.8 MB
    valid_labels.parquet           15.2 MB


In [ ]:
import polars as pl
from pathlib import Path

OUTPUT_DIR = Path(__file__).parent.parent / 'datasets' / 'otto-train-val'
TRAIN_SESSIONS_PARQUET = OUTPUT_DIR / 'train_sessions.parquet'
VALID_PARQUET          = OUTPUT_DIR / 'valid_inputs.parquet'
VALID_LABELS_PARQUET   = OUTPUT_DIR / 'valid_labels.parquet'

train  = pl.read_parquet(TRAIN_SESSIONS_PARQUET)
inputs = pl.read_parquet(VALID_PARQUET)
labels = pl.read_parquet(VALID_LABELS_PARQUET)

print("=" * 60)
print("1. SCHEMA & SHAPE")
print("=" * 60)
print(f"\ntrain_sessions : {train.shape}")
print(train.schema)
print(f"\nvalid_inputs   : {inputs.shape}")
print(inputs.schema)
print(f"\nvalid_labels   : {labels.shape}")
print(labels.schema)

print("\n" + "=" * 60)
print("2. SAMPLE DATA")
print("=" * 60)
print("\n-- train (1 row) --")
print(train.head(1))
print("\n-- valid_inputs (1 row) --")
print(inputs.head(1))
print("\n-- valid_labels (1 row) --")
print(labels.head(1))

print("\n" + "=" * 60)
print("3. NULL CHECK")
print("=" * 60)
print(f"\ntrain  nulls: {train.null_count().row(0)}")
print(f"inputs nulls: {inputs.null_count().row(0)}")
print(f"labels nulls: {labels.null_count().row(0)}")

print("\n" + "=" * 60)
print("4. SESSION ID CHECKS")
print("=" * 60)

input_sessions = set(inputs['session'].to_list())
label_sessions = set(labels['session'].to_list())
train_session_ids = set(train['session'].to_list())

print(f"\ninputs sessions : {len(input_sessions):,}")
print(f"labels sessions : {len(label_sessions):,}")
print(f"inputs == labels: {input_sessions == label_sessions}")

overlap = input_sessions & train_session_ids
print(f"\nTrain/valid session overlap: {len(overlap):,}  (phải = 0)")

print("\n" + "=" * 60)
print("5. EVENTS SANITY CHECK")
print("=" * 60)

train_event_lens = train.with_columns(
    pl.col('events').list.len().alias('n_events')
)['n_events']
input_event_lens = inputs.with_columns(
    pl.col('events').list.len().alias('n_events')
)['n_events']

print(f"\ntrain  events/session — min:{train_event_lens.min()}, mean:{train_event_lens.mean():.1f}, max:{train_event_lens.max()}")
print(f"inputs events/session — min:{input_event_lens.min()}, mean:{input_event_lens.mean():.1f}, max:{input_event_lens.max()}")
print(f"Any session with 0 events (train) : {(train_event_lens == 0).sum()}")
print(f"Any session with 0 events (inputs): {(input_event_lens == 0).sum()}")

print("\n" + "=" * 60)
print("6. LABEL SANITY CHECK")
print("=" * 60)

label_df = labels.with_columns([
    pl.col('labels').struct.field('clicks').alias('clicks'),
    pl.col('labels').struct.field('carts').alias('carts'),
    pl.col('labels').struct.field('orders').alias('orders'),
])

n_has_clicks = label_df['clicks'].drop_nulls().len()
n_has_carts  = label_df.filter(pl.col('carts').list.len() > 0).height
n_has_orders = label_df.filter(pl.col('orders').list.len() > 0).height
total        = labels.height

print(f"\nSessions có clicks label : {n_has_clicks:,} ({100*n_has_clicks/total:.1f}%)")
print(f"Sessions có carts  label : {n_has_carts:,}  ({100*n_has_carts/total:.1f}%)")
print(f"Sessions có orders label : {n_has_orders:,} ({100*n_has_orders/total:.1f}%)")
print(f"Sessions không có label nào: "
      f"{label_df.filter(pl.col('clicks').is_null() & (pl.col('carts').list.len()==0) & (pl.col('orders').list.len()==0)).height:,}")

print("\n" + "=" * 60)
print("7. ITEM LEAKAGE CHECK")
print("=" * 60)

# So sánh với train_sessions — đúng nguồn model được train
train_aids = set(
    train
    .explode('events')
    .unnest('events')
    ['aid'].to_list()
)
valid_aids = set(
    inputs
    .explode('events')
    .unnest('events')
    ['aid'].to_list()
)

leaked = valid_aids - train_aids
print(f"\nTrain unique aids : {len(train_aids):,}")
print(f"Valid unique aids  : {len(valid_aids):,}")
print(f"Unknown aids in valid (phải = 0): {len(leaked):,}")

print("\n" + "=" * 60)
print("8. TIMESTAMP ORDER CHECK (sample 1000 sessions)")
print("=" * 60)

sample = inputs.sample(min(1000, inputs.height), seed=42)
n_unsorted = 0
for row in sample.iter_rows(named=True):
    ts_list = [e['ts'] for e in row['events']]
    if ts_list != sorted(ts_list):
        n_unsorted += 1

print(f"\nSessions có events không theo thứ tự ts: {n_unsorted} (phải = 0)")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
checks = {
    "inputs == labels session ids" : input_sessions == label_sessions,
    "No train/valid overlap"       : len(overlap) == 0,
    "No 0-event sessions (train)"  : (train_event_lens == 0).sum() == 0,
    "No 0-event sessions (inputs)" : (input_event_lens == 0).sum() == 0,
    "No unknown aids in valid"     : len(leaked) == 0,
    "Events sorted by ts"          : n_unsorted == 0,
}
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check}")

1. SCHEMA & SHAPE

train_sessions : (10584517, 2)
Schema({'session': Int64, 'events': List(Struct({'aid': Int64, 'ts': Int64, 'type': String}))})

valid_inputs   : (1783790, 2)
Schema({'session': Int64, 'events': List(Struct({'aid': Int64, 'ts': Int64, 'type': String}))})

valid_labels   : (1783790, 2)
Schema({'session': Int64, 'labels': Struct({'clicks': Int64, 'carts': List(Int64), 'orders': List(Int64)})})

2. SAMPLE DATA

-- train (1 row) --
shape: (1, 2)
┌─────────┬─────────────────────────────────┐
│ session ┆ events                          │
│ ---     ┆ ---                             │
│ i64     ┆ list[struct[3]]                 │
╞═════════╪═════════════════════════════════╡
│ 8750975 ┆ [{338668,1660554578918,"clicks… │
└─────────┴─────────────────────────────────┘

-- valid_inputs (1 row) --
shape: (1, 2)
┌──────────┬─────────────────────────────────┐
│ session  ┆ events                          │
│ ---      ┆ ---                             │
│ i64      ┆ list[struct[3]]   